# Prepare dashboard data: Metro Interstate Traffic Volume

This notebook downloads the UCI dataset, cleans it and adds columns that are handy for a Power BI dashboard (year, month, hour, weekday/weekend, time of day, congestion level). It saves **traffic_dashboard_data.csv**.

Run all cells, then download the CSV from the Files panel (folder icon on the left).

In [ ]:
import io, zipfile, urllib.request
import numpy as np
import pandas as pd

URL = "https://archive.ics.uci.edu/static/public/492/metro+interstate+traffic+volume.zip"

def load_data():
    try:
        with urllib.request.urlopen(URL, timeout=60) as r:
            z = zipfile.ZipFile(io.BytesIO(r.read()))
        name = [n for n in z.namelist() if n.endswith(".csv") or n.endswith(".csv.gz")][0]
        return pd.read_csv(io.BytesIO(z.read(name)), compression="gzip" if name.endswith(".gz") else None)
    except Exception as e:
        print("Automatic download failed:", e)
        from google.colab import files
        up = files.upload()
        return pd.read_csv(list(up.keys())[0])

raw = load_data()
print(raw.shape)

In [ ]:
df = raw.copy()
df["date_time"] = pd.to_datetime(df["date_time"])
df = df.drop_duplicates(subset="date_time", keep="first").sort_values("date_time").reset_index(drop=True)

# cleaning
df["temp_c"] = (df["temp"] - 273.15).round(1)
df.loc[df["temp"] < 200, "temp_c"] = np.nan          # impossible 0 K readings
df.loc[df["rain_1h"] > 100, "rain_1h"] = np.nan       # extreme rainfall outlier
df["holiday"] = df["holiday"].fillna("None")

# columns for the dashboard
df["Date"] = df["date_time"].dt.date
df["Year"] = df["date_time"].dt.year
df["Month_Number"] = df["date_time"].dt.month
df["Month"] = df["date_time"].dt.strftime("%b")
df["Day_Of_Week"] = df["date_time"].dt.day_name()
df["Day_Number"] = df["date_time"].dt.dayofweek + 1   # Monday = 1 (used to sort the weekdays)
df["Hour"] = df["date_time"].dt.hour
df["Day_Type"] = np.where(df["date_time"].dt.dayofweek >= 5, "Weekend", "Weekday")
df["Time_Of_Day"] = pd.cut(df["Hour"], bins=[-1, 5, 9, 15, 19, 23],
                           labels=["Night (0-5)", "Morning peak (6-9)", "Midday (10-15)", "Evening peak (16-19)", "Late evening (20-23)"])
df["Time_Of_Day_Order"] = pd.cut(df["Hour"], bins=[-1, 5, 9, 15, 19, 23], labels=[1, 2, 3, 4, 5]).astype(int)   # used to sort Time_Of_Day
df["Is_Holiday"] = np.where(df["holiday"] == "None", "No", "Yes")

q1, q2 = df["traffic_volume"].quantile([0.33, 0.66])
df["Congestion_Level"] = pd.cut(df["traffic_volume"], bins=[-1, q1, q2, np.inf], labels=["Low", "Medium", "High"])
print(f"Low <= {q1:.0f} < Medium <= {q2:.0f} < High")

out = df[["date_time", "Date", "Year", "Month_Number", "Month", "Day_Of_Week", "Day_Number", "Hour",
          "Day_Type", "Time_Of_Day", "Time_Of_Day_Order", "Is_Holiday", "holiday", "traffic_volume", "Congestion_Level",
          "temp_c", "rain_1h", "snow_1h", "clouds_all", "weather_main"]].copy()
out = out.rename(columns={"date_time": "Date_Time", "holiday": "Holiday_Name", "traffic_volume": "Traffic_Volume",
                          "temp_c": "Temp_C", "rain_1h": "Rain_mm", "snow_1h": "Snow_mm",
                          "clouds_all": "Cloud_Cover_Pct", "weather_main": "Weather"})
out.to_csv("traffic_dashboard_data.csv", index=False)
print(out.shape)
out.head()

In [ ]:
from google.colab import files
files.download("traffic_dashboard_data.csv")